In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/processing/task1_ielts_dataset.csv')

print("Dataset shape:", df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
print("Missing values:\n")
print(df.isnull().sum())

print("\nOverall missing:", df["Overall"].isnull().sum())

In [ ]:
df.nunique()

In [ ]:
duplicate_essay = df["Essay"].duplicated().sum()
print("Duplicate essays:", duplicate_essay)

In [ ]:
essay_multi_question = (
    df.groupby("Essay")["Question"]
    .nunique()
    .value_counts()
)

print("Essay -> number of unique Questions:\n")
print(essay_multi_question)

In [ ]:
essay_multi_score = (
    df.groupby("Essay")["Overall"]
    .nunique()
    .value_counts()
)

print("Essay -> number of unique Overall scores:\n")
print(essay_multi_score)

In [ ]:
print("Unique Overall scores:")
print(sorted(df["Overall"].unique()))

Raw dataset may contain missing values in criteria columns,
which will be handled during preprocessing.

In [ ]:
bins = np.arange(1, 9.5, 0.5)

hist = plt.hist(
    df["Overall"],
    bins=bins,
    edgecolor='black',
    alpha=0.7
)

plt.title("Distribution of IELTS Scores")
plt.xlabel("IELTS Scores")
plt.ylabel("Frequency")

heights = hist[0]
edges = hist[1]

for i in range(len(heights)):
    x = (edges[i] + edges[i+1]) / 2
    y = heights[i]

    if y > 0:
        plt.text(x, y, str(int(y)), ha='center', va='bottom')

plt.xticks(np.arange(1, 9.5, 0.5))

plt.show()

In [ ]:
score_counts = df["Overall"].value_counts().sort_index()

score_counts

score_counts.plot(kind="bar")

plt.title("Score Class Distribution")
plt.xlabel("Band Score")
plt.ylabel("Number of Essays")

plt.show()


# This analysis identifies whether the dataset suffers from class imbalance.

In [ ]:
print("Skewness:", df["Overall"].skew())
print("Kurtosis:", df["Overall"].kurtosis())

In [ ]:
cumulative_freq = (
    df
    .groupby("Overall")
    .agg(Count=("Overall", "count"))
    .sort_index()
)

cumulative_freq["Cumulative_Frequency"] = cumulative_freq["Count"].cumsum()

total_count = cumulative_freq["Count"].sum()

cumulative_freq["Percentage"] = round(
    (cumulative_freq["Cumulative_Frequency"] / total_count) * 100,
    3
)

cumulative_freq

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    cumulative_freq.index,
    cumulative_freq["Percentage"],
    marker='o'
)

plt.title("Cumulative Percentage of IELTS Scores")
plt.xlabel("Band Score")
plt.ylabel("Percentage (%)")

plt.xticks(np.arange(1, 9.5, 0.5))

plt.grid()

plt.show()

In [ ]:
df["length"] = df["Essay"].str.split().str.len()

df["length"].describe()

In [ ]:
plt.hist(
    df["length"],
    bins=50,
    edgecolor="black"
)

plt.title("Essay Length Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")

plt.show()

In [ ]:
length_by_score = (
    df
    .groupby("Overall")
    .agg(mean_length=("length","mean"))
    .reset_index()
)

length_by_score

In [ ]:
plt.plot(
    length_by_score["Overall"],
    length_by_score["mean_length"],
    marker='o'
)

plt.title("Average Essay Length by Band Score")
plt.xlabel("Band Score")
plt.ylabel("Average Word Count")

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    df["Overall"],
    df["length"],
    alpha=0.5
)

a, b = np.polyfit(df["Overall"], df["length"], 1)

x_line = np.linspace(
    df["Overall"].min(),
    df["Overall"].max(),
    100
)

y_line = a * x_line + b

plt.plot(
    x_line,
    y_line,
    color='red',
    label=f'y = {a:.2f}x + {b:.2f}'
)

plt.title("Essay Length vs Overall Score")
plt.xlabel("Overall Score")
plt.ylabel("Essay Length")

plt.xticks(np.arange(1, 9.5, 0.5))

plt.grid()
plt.legend()

plt.show()

In [ ]:
score_variance = (
    df
    .groupby("Overall")
    .agg(
        mean_length=("length","mean"),
        std_length=("length","std"),
        count=("length","count")
    )
)

score_variance

In [ ]:
score_variance = (
    df
    .groupby("Overall")
    .agg(
        mean_length=("length","mean"),
        std_length=("length","std"),
        count=("length","count")
    )
)

score_variance

In [ ]:
plt.errorbar(
    score_variance.index,
    score_variance["mean_length"],
    yerr=score_variance["std_length"],
    fmt='o'
)

plt.title("Score Variance Analysis")
plt.xlabel("Band Score")
plt.ylabel("Essay Length (Mean ± Std)")

plt.grid()
plt.show()
# This analysis helps understand the variance of essays by score.


In [ ]:
print("Very short essays:")
df[df["length"] < 100][["Essay","Overall"]].head()

print("\nVery long essays:")
df[df["length"] > 300][["Essay","Overall"]].head()

In [ ]:
criteria = [
    "Task_Achievement",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]

In [ ]:
for col in criteria:
    plt.hist(df[col], bins=np.arange(1,9.5,0.5), edgecolor="black")
    plt.title(f"{col} Distribution")
    plt.xlabel("Score")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
corr = df[criteria + ["Overall"]].corr()

corr

In [ ]:
plt.imshow(corr, cmap='coolwarm', interpolation='none')
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha='center', va='center',
            color='black'
        )

plt.title("Correlation Matrix")
plt.show()

In [ ]:
def round_band(x):
    if pd.isna(x):
        return np.nan

    integer = int(x)
    decimal = x - integer

    if decimal < 0.25:
        return integer
    elif decimal <= 0.5:
        return integer + 0.5
    elif decimal < 0.75:
        return integer + 0.5
    else:
        return integer + 1

In [ ]:
df["criteria_mean"] = df[criteria].mean(axis=1).apply(round_band)

diff = (df["criteria_mean"] - df["Overall"]).abs()

print("Mean difference:", diff.mean())
print("Max difference:", diff.max())

In [ ]:
plt.hist(diff, bins=50, edgecolor="black")

plt.title("Difference between Mean Criteria and Overall")
plt.xlabel("Absolute Difference")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print("Number of unique questions:", df["Question"].nunique())

In [ ]:
question_counts = df["Question"].value_counts()

question_counts.head()

In [ ]:
question_counts.plot(kind="hist", bins=50)

plt.title("Number of Essays per Question")
plt.xlabel("Count")
plt.ylabel("Frequency")

plt.show()